In [851]:
# Data read

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

time_list = []

# train_1: 배추,무,양파,감자,대파 
# train_2: 건고추, 깐마늘, 상추, 사과, 배
# train_3: 기상데이터


train1 = pd.read_csv('Data_2nd/train_1.csv')
train2 = pd.read_csv('Data_2nd/train_2.csv')
train3 = pd.read_csv('Data_2nd/TRAIN_기상_2018-2022.csv')




for i in range(len(train1['YYYYMMSOON'])):
    time_list.append(train1['YYYYMMSOON'][i])
timelist = np.unique(time_list)

data_case = []

for i in range(len(train1['품목(품종)명'])):
    data_case.append(train1['품목(품종)명'][i])
    
    
for i in range(len(train2['품목명'])):
    data_case.append(train2['품목명'][i])
    
data_case = np.unique(data_case)
data_case

array(['감자 수미', '건고추', '깐마늘(국산)', '대파(일반)', '무', '배', '배추', '사과', '상추',
       '양파'], dtype='<U7')

In [852]:
train3[90:100]

,YYYYMMSOON,지역 이름,주산지 품목명,주산지 품종명,순 평균상대습도,순 평균기온,순 평균풍속,순 최고기온,순 최저상대습도,순 최저기온,순 강수량,순 누적 일조시간,특보 코드,특보 명,특보 발효 순통계 개수
90,201801중순,N,배추,가을,0.898990,0.261905,0.142857,0.275,0.292135,0.156863,0.045638,0.000000,S3,대설경보,0.090909
91,201801중순,A,배추,가을,0.676768,0.214286,0.142857,0.200,0.235955,0.117647,0.000000,0.358779,C2,한파주의보,0.181818
92,201801중순,R,배추,고랭지,0.585859,0.190476,0.142857,0.150,0.168539,0.137255,0.008054,0.328244,C3,한파경보,0.181818
93,201801중순,R,배추,고랭지,0.585859,0.190476,0.142857,0.150,0.168539,0.137255,0.008054,0.328244,D2,건조주의보,0.545455
94,201801중순,B,배추,가을,0.787879,0.238095,0.142857,0.175,0.258427,0.235294,0.001342,0.000000,C2,한파주의보,0.181818
95,201801중순,H,배추,가을,0.717172,0.190476,0.142857,0.200,0.247191,0.078431,0.004027,0.335878,C3,한파경보,0.181818
96,201801중순,H,배추,가을,0.717172,0.190476,0.142857,0.200,0.247191,0.078431,0.004027,0.335878,C2,한파주의보,0.181818
97,201801중순,A,배추,가을,0.676768,0.214286,0.142857,0.200,0.235955,0.117647,0.000000,0.358779,C3,한파경보,0.181818
98,201801중순,Q,배추,고랭지,0.424242,0.309524,0.142857,0.325,0.112360,0.274510,0.006711,0.000000,D2,건조주의보,0.272727
99,201801중순,Q,배추,고랭지,0.424242,0.309524,0.142857,0.325,0.112360,0.274510,0.006711,0.000000,D3,건조경보,0.454545


In [853]:
train3['순 평균상대습도']

0        0.616162
1        0.737374
2        0.535354
3        0.656566
4        0.646465
           ...   
46515    0.696970
46516    0.707071
46517    0.707071
46518    0.595960
46519    0.818182
Name: 순 평균상대습도, Length: 46520, dtype: float64

In [854]:
from os import path
from itertools import product
from copy import deepcopy

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

CASE = [
    "배추",
    "무",
    "양파",
    "감자",
    "대파(일반)",
    "건고추",
    "깐마늘(국산)",
    "상추",
    "사과",
    "배",
]

weather_subcases = ['순 평균상대습도', '순 평균기온', '순 평균풍속', '순 최고기온', '순 최저상대습도', '순 최저기온',
                  '순 강수량', '순 누적 일조시간', '특보 발효 순통계 개수']

# sub_case:(주산지 품좀명,   지역이름)을 품목명에 대해 알아보자


sub_cases1 = {}

for case in CASE:
    sub_case1 = []
     
    for n in range(len(train3)):
        if case == train3['주산지 품목명'][n]:
            sub_case1.append(train3['주산지 품종명'][n])
            
    sub_cases1[case] = np.unique(sub_case1)

sub_cases2 = {}
for case in CASE:
    detailed_sub_case2 = {}
    for subcase in sub_cases1[case]:
        sub_case2 = []
        for n in range(len(train3)):
            if case == train3['주산지 품목명'][n]:
                if subcase == train3['주산지 품종명'][n]:
                    sub_case2.append(train3['지역 이름'][n])
            
        detailed_sub_case2[subcase] = np.unique(sub_case2)
    sub_cases2[case] = detailed_sub_case2
    
    
# weather_data = {}  



a = 0
for case in CASE:
    sub_data = {}
    for sub_case1 in sub_cases1[case]:
        weather_sub_data1 = {}
        for sub_case2 in sub_cases2[case][sub_case1]:
            weather_sub_data2 = {}
            for weather_subcase in weather_subcases:
                print('case = {}, sub_case1 = {}, sub_case2 = {}, weather_case = {}'.format(
                case, sub_case1, sub_case2, weather_subcase))
                if a % 45 == 0:
                    print(a/2250)
                weather_subdata = []
                
                
                
                for time in timelist:
     
                    filtered_train = train3[
                        (train3['주산지 품목명'] == case) & 
                        (train3['YYYYMMSOON'] == time) &
                        (train3['주산지 품종명'] == sub_case1) &
                        (train3['지역 이름'] == sub_case2) 
                    ][['주산지 품목명', '주산지 품종명', '지역 이름', 'YYYYMMSOON'] + [weather_subcase]]
                    filtered_train = filtered_train.groupby(['YYYYMMSOON', '주산지 품목명', '주산지 품종명', '지역 이름'], as_index=False).mean()

                    
                    #                     print(filtered_train['순 평균상대습도'][0])
                    
    
                    if len(filtered_train) == 0:
                        break
                
                    weather_subdata.append(filtered_train[weather_subcase][0])
                a += 1
                print(len(weather_subdata))
                if len(weather_subdata) == 180 and not any(np.isnan(value) for value in weather_subdata):
                    weather_sub_data2[weather_subcase] = weather_subdata
            if weather_sub_data2:
                weather_sub_data1[sub_case2] = weather_sub_data2
        sub_data[sub_case1] = weather_sub_data1
    weather_data[case] = sub_data

case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 순 평균상대습도
0.0
180
case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 순 평균기온
180
case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 순 평균풍속
180
case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 순 최고기온
180
case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 순 최저상대습도
180
case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 순 최저기온
180
case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 순 강수량
180
case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 순 누적 일조시간
180
case = 배추, sub_case1 = 가을, sub_case2 = A, weather_case = 특보 발효 순통계 개수
180
case = 배추, sub_case1 = 가을, sub_case2 = AA, weather_case = 순 평균상대습도
0
case = 배추, sub_case1 = 가을, sub_case2 = AA, weather_case = 순 평균기온
0
case = 배추, sub_case1 = 가을, sub_case2 = AA, weather_case = 순 평균풍속
0
case = 배추, sub_case1 = 가을, sub_case2 = AA, weather_case = 순 최고기온
0
case = 배추, sub_case1 = 가을, sub_case2 = AA, weather_case = 순 최저상대습도
0
case = 배추, sub_case1 = 가을, sub_ca

In [855]:
sub_cases1['건고추'][0] == '-'

True

In [856]:
filtered_train = train3[
                        (train3['주산지 품목명'] == '사과') 
                        
                    ]

In [857]:
filtered_train

,YYYYMMSOON,지역 이름,주산지 품목명,주산지 품종명,순 평균상대습도,순 평균기온,순 평균풍속,순 최고기온,순 최저상대습도,순 최저기온,순 강수량,순 누적 일조시간,특보 코드,특보 명,특보 발효 순통계 개수
38,201801상순,S,사과,-,0.595960,0.238095,0.142857,0.175,0.191011,0.294118,0.006711,0.404580,00,특보없음,0.000000
39,201801상순,L,사과,-,0.545455,0.214286,0.142857,0.175,0.134831,0.254902,0.001342,0.435115,00,특보없음,0.000000
103,201801중순,L,사과,-,0.696970,0.214286,0.142857,0.300,0.202247,0.117647,0.016107,0.427481,C3,한파경보,0.363636
104,201801중순,S,사과,-,0.737374,0.238095,0.142857,0.300,0.213483,0.215686,0.016107,0.458015,C2,한파주의보,0.272727
192,201801하순,S,사과,-,0.494949,0.166667,0.142857,0.225,0.134831,0.156863,0.000000,0.664122,C2,한파주의보,0.454545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46494,202212하순,O,사과,-,0.595960,0.214286,0.285714,0.100,0.359551,0.235294,0.009396,0.519084,S2,대설주의보,0.090909
46495,202212하순,O,사과,-,0.595960,0.214286,0.285714,0.100,0.359551,0.235294,0.009396,0.519084,T2,태풍주의보,1.000000
46496,202212하순,AZ,사과,-,0.595960,0.190476,0.285714,0.100,0.258427,0.215686,0.006711,0.465649,T2,태풍주의보,1.000000
46497,202212하순,W,사과,-,0.696970,0.119048,0.142857,0.100,0.280899,0.156863,0.006711,0.572519,00,특보없음,0.000000


In [858]:
# print(data_case[0])
# print(sub_cases1[data_case[0]])
weather_data['배추']['가을']['A']['순 평균상대습도']

[0.6161616161616162,
 0.6767676767676768,
 0.5555555555555556,
 0.5555555555555556,
 0.4343434343434343,
 0.6363636363636365,
 0.6767676767676768,
 0.6666666666666667,
 0.5757575757575758,
 0.696969696969697,
 0.5656565656565657,
 0.6161616161616162,
 0.6767676767676768,
 0.7777777777777778,
 0.6464646464646465,
 0.6363636363636365,
 0.6666666666666667,
 0.7171717171717172,
 0.8383838383838385,
 0.8181818181818182,
 0.7272727272727273,
 0.7474747474747475,
 0.7272727272727272,
 0.8282828282828284,
 0.7474747474747475,
 0.787878787878788,
 0.7474747474747475,
 0.7777777777777778,
 0.7676767676767677,
 0.7676767676767677,
 0.787878787878788,
 0.7676767676767677,
 0.7676767676767677,
 0.6363636363636365,
 0.7272727272727273,
 0.5656565656565657,
 0.5757575757575758,
 0.6363636363636365,
 0.5555555555555556,
 0.5656565656565657,
 0.6161616161616162,
 0.6161616161616162,
 0.5656565656565657,
 0.6565656565656566,
 0.6363636363636365,
 0.5151515151515152,
 0.5656565656565657,
 0.6666666666666

In [859]:
# train_data1: 배추~배의 평년평균가격

train_data1 = {}

data1 = np.asarray(train1['평균가격(원)']).reshape(5,180)
data2 = np.asarray(train2['평균가격(원)']).reshape(5,180)



for case_n,n in enumerate([0,180,360,540,720]):
    train_data1[train1.loc[n]['품목(품종)명']] = data1[case_n]
    
    

for case_n,n in enumerate([0,180,360,540,720]):
    train_data1[train2.loc[n]['품목명']] = data2[case_n]

In [860]:
# 결측치 확인
anomaly_cases = []
for case in data_case:
    anomaly_case = 0
    for n in range(180):
        if train_data1[case][n] == 0:
            anomaly_case += 1

    if anomaly_case > 1:
        anomaly_cases.append(case)
        
print(anomaly_cases)

[]


In [861]:
train_data1['건고추'][175:180]

array([650267., 639100., 630600., 626400., 626400.])

In [862]:
def data_seperate(data):
    train_x = []
    train_y = []
    
    T = len(data) -9
    for t in range(T):
        x = []
        for n in range(9):
            x.append(data[t+n])
     
        train_x.append(x)
        train_y.append(data[t+9])
        
        
    return np.asarray(train_x).reshape(len(train_x),9), np.asarray(train_y).reshape(len(train_y),1)

In [863]:
def nmae(y,yhat):
    loss = np.zeros(len(y))
    for n in range(len(y)):
        loss[n] = (np.abs(y[n] - yhat[n])/y[n])
        
    return np.mean(loss)

In [864]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

 
from sklearn.linear_model import LinearRegression

In [865]:
def get_train_val_splits(data, interval=36):

    train_datas = []
    val_datas = []
    data = np.asarray(data)
    # 전체 데이터를 interval 단위로 반복하여 분할
    for start in range(0, 180, interval):
        end = start + interval
        # validation 데이터 설정
        val_data = data[start:end]
        
        # train 데이터 설정 (validation 구간을 제외한 나머지 구간)
        train_data = np.concatenate((data[:start], data[end:]), axis=0)
        
        # 결과 저장
        train_datas.append(train_data)
        val_datas.append(val_data)
    return train_datas, val_datas

In [866]:
data1 = train_data1['배추']

for n in range(5):
    print(len(get_train_val_splits(data1, interval=36)[0][n]))
    print(len(get_train_val_splits(data1, interval=36)[1][n]))

144
36
144
36
144
36
144
36
144
36


In [867]:
#  이 함수는 가격과 피처 하나만 활용가능

def validation(data1,data2, d_n):
#     data1 = np.asarray(data1)
    
#     data2 = np.asarray(data2)
    d_n = d_n
    
    train_data1, val_data1 = get_train_val_splits(data1, interval=36)
    train_data2, val_data2 = get_train_val_splits(data2, interval=36)
    
    val_loss = np.zeros(5)
    train_loss = np.zeros(5)
    
    if d_n > 1:
        for v_n in range(5):

            train_x1,train_y1 = data_seperate(train_data1[v_n])
            train_x2,train_y2 = data_seperate(train_data2[v_n])

            val_x1,val_y1 = data_seperate(val_data1[v_n])
            val_x2,val_y2 = data_seperate(train_data2[v_n])


            train_x = np.zeros((len(train_x1),9*d_n))
            val_x = np.zeros((len(val_x1),9*d_n))

            for n in range(len(train_x1)):
                a = 0
                for t in range(0,18,2):
                    train_x[n,t] = train_x1[n,a]
                    train_x[n,t+1] = train_x2[n,a]

                    a += 1

            for n in range(len(val_x1)):
                a = 0
                for t in range(0,18,2):
                    val_x[n,t] = val_x1[n,a]
                    val_x[n,t+1] = val_x2[n,a]

                    a += 1


            linear_regression = LinearRegression()


            linear_regression.fit(train_x, train_y1)


            pred_train_y = np.zeros((len(train_x),1))

            for n in range(len(train_x)):
                pred_train_y[n,0] = linear_regression.predict(train_x[n].reshape(1,9*d_n))[0]

            train_loss[v_n] = (nmae(train_y1, pred_train_y))


            pred_val_y = np.zeros((len(val_x),1))
            for n in range(len(val_x)):
                pred_val_y[n,0] = linear_regression.predict(val_x[n].reshape(1,9*d_n))[0]

            val_loss[v_n] = (nmae(val_y1, pred_val_y))
            
            
    elif d_n == 1:
        for v_n in range(5):

            train_x1,train_y1 = data_seperate(train_data1[v_n])
            val_x1,val_y1 = data_seperate(val_data1[v_n])

            train_x = np.zeros((len(train_x1),9*d_n))
            val_x = np.zeros((len(val_x1),9*d_n))

            for n in range(len(train_x1)):
                a = 0
                for t in range(0,9,1):
                    train_x[n,t] = train_x1[n,a]


                    a += 1

            for n in range(len(val_x1)):
                a = 0
                for t in range(0,9,1):
                    val_x[n,t] = val_x1[n,a]


                    a += 1

            linear_regression = LinearRegression()


            linear_regression.fit(train_x, train_y1)


            pred_train_y = np.zeros((len(train_x),1))

            for n in range(len(train_x)):
                pred_train_y[n,0] = linear_regression.predict(train_x[n].reshape(1,9*d_n))[0]

            train_loss[v_n] = (nmae(train_y1, pred_train_y))


            pred_val_y = np.zeros((len(val_x),1))
            for n in range(len(val_x)):
                pred_val_y[n,0] = linear_regression.predict(val_x[n].reshape(1,9*d_n))[0]

            val_loss[v_n] = (nmae(val_y1, pred_val_y))
    return np.min(train_loss), np.min(val_loss), np.argmax(val_loss)

In [868]:
# for evaluate(train_data, weather_data):

data1 = train_data1['배추']
data2 = weather_data['배추']['가을']['B'][weather_subcases[0]]

validation(data1,data2, 1)

(0.1334460411863271, 0.1533623381023711, 1)

In [869]:
sub_cases2['배추']

{'가을': array(['A', 'AA', 'AG', 'AM', 'AR', 'AW', 'B', 'BB', 'BN', 'BQ', 'BR',
        'BU', 'BX', 'CH', 'CN', 'CO', 'CP', 'CT', 'E', 'H', 'N', 'X', 'Z'],
       dtype='<U2'),
 '겨울': array(['AA', 'AF', 'AG', 'AM', 'BB', 'D'], dtype='<U2'),
 '고랭지': array(['AS', 'AU', 'AV', 'AX', 'CM', 'Q', 'R'], dtype='<U2'),
 '봄': array(['AG', 'AY', 'AZ', 'BA', 'BB', 'BE', 'BN', 'BQ', 'BR', 'BU', 'BX',
        'CI', 'CJ', 'CK', 'CL', 'CQ', 'CR', 'CS', 'CU', 'CV', 'CW', 'CX',
        'G', 'N', 'W', 'Z'], dtype='<U2')}

In [870]:
list(weather_data['배추']['가을'].keys())

['A', 'B', 'H', 'N']

In [871]:
data_case

array(['감자 수미', '건고추', '깐마늘(국산)', '대파(일반)', '무', '배', '배추', '사과', '상추',
       '양파'], dtype='<U7')

In [872]:
len({})

0

In [873]:
# 기상데이터가 없거나 결측치가 있는 거 제외

for case in data_case[1:]:
    print(case)
    print(len(weather_data[case]))

건고추
1
깐마늘(국산)
0
대파(일반)
0
무
4
배
1
배추
4
사과
1
상추
0
양파
2


In [874]:
weather_data['사과']

{'-': {}}

In [875]:

train_loss = []
val_loss = []

good_params = []

measurement = []

case = '양파'
for subcase in list(weather_data[case].keys()):
    for area in list(weather_data[case][subcase].keys()):
        for w_case in list(weather_data[case][subcase][area].keys()):

            data1 = train_data1[case]
            data2 = weather_data[case][subcase][area][w_case]


            val_result = validation(data1,data2, 2)

            train_loss.append(val_result[0])


            val_loss.append(val_result[1])

            good_params.append(val_result[2])
            
            measurement.append([subcase, area, w_case])
            
            

In [876]:
print(np.argmin(val_loss))
print(train_loss[np.argmin(val_loss)])
print(val_loss[np.argmin(val_loss)])

1
0.08714136347854051
0.07791072944096913


In [877]:
print(measurement[np.argmin(val_loss)])

['중만생종', 'K', '순 평균기온']


10-26 결과 (기상데이터를 선형회귀에 적용해봄, 가격과 각각의 피처 하나씩만 넣어서 해봤을 때 성능 개선확인)

배추랑 가장 잘맞는 피처는 순 강수량!! (지역 A에서)

print(weather_subcases[np.argmin(val_loss)])

가격만 했을때, train = 0.133, val = 0.153

가격 + 순 강수량(A), train = 0.130, val = 0.137 -- best

가격 + 순 강수량(B), train = 0.134, val = 0.139

가격 + 순 최고기온(H), train = 0.133, val = 0.143

가격 + 순 최저기온(N), train = 0.136, val = 0.153

....

배추는 가격 + ['가을', 'A', '순 강수량'], train = 0.130, val = 0.137 -- best

건고추는 가격, train = 0.021, val = 0.020 -- best

무는 가격 + ['월동', 'D', '순 평균풍속'], train = 0.123, val = 0.102 -- best

양파는 가격 + ['중만생종', 'K', '순 평균기온'], train = 0.087, val = 0.078 -- best

나머지는 기상데이터가 없거나 결측치가 있음.

10-27 오후에 피처를 다 넣어보고 결과 확인 + DL modeling 예정

In [878]:
data1 = train_data1['배추']
data2 = weather_data['배추']['가을']['A'][weather_subcases[0]]
data3 = weather_data['배추']['가을']['A'][weather_subcases[1]]
data4 = weather_data['배추']['가을']['A'][weather_subcases[2]]
data5 = weather_data['배추']['가을']['A'][weather_subcases[3]]
data6 = weather_data['배추']['가을']['A'][weather_subcases[4]]
data7 = weather_data['배추']['가을']['A'][weather_subcases[5]]
data8 = weather_data['배추']['가을']['A'][weather_subcases[6]]
data9 = weather_data['배추']['가을']['A'][weather_subcases[7]]

In [879]:
train_datas = []
val_datas = []

for data in [data1,data2,data3,data4,data5,data6,data7,data8,data9]:
    train_data, val_data = get_train_val_splits(data, interval=36)
    
    
    train_datas.append(train_data)
    val_datas.append(val_data)

.... 피처 다 해보는 건 내일.